# 1 · Model

Notebook 0's model is three modules glued together with `nn.Sequential`. That is fine until it has to change:

- The channel count 16 is written in all three parts; widening the encoder means editing the other two to match.
- Trying another encoder means rewriting the glue.

This notebook gives each part a declared interface so the parts can change one at a time. We also register our own model parts, load a small pretrained model from the built-in catalogue, add augmentation, and compare the runs in TensorBoard. The task and the data stay the same.

The same data, opened the same way. If notebook 0 already fetched it, the download reuses those files.

In [ ]:
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from torch import nn
from torch.utils.data import DataLoader, Dataset

import geosave_engine as gs

root = Path(snapshot_download("fatmur/dynamicworld-300", repo_type="dataset", local_dir="data/dynamic_world"))


class Tiles(Dataset):
    def __init__(self, split):
        self.folder = root / split
        self.names = sorted(path.name for path in (self.folder / "images").iterdir())

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        name = self.names[index]
        image = gs.open_raster(self.folder / "images" / name).isel(time=0)
        label = gs.open_raster(self.folder / "labels" / name).isel(time=0)
        return image.gs.to_tensor(dtype="float32") / 10_000, label.gs.to_tensor(dtype="int64")[0]


train = Tiles("train")
val = Tiles("val")
train_loader = DataLoader(train, batch_size=8, shuffle=True)
val_loader = DataLoader(val, batch_size=8)

## The same three parts

These are still ordinary PyTorch modules: an encoder, a decoder, and a head. We add declarations to let GeoSave connect them and build them by name; their layers and direct calls work as before.

In [ ]:
from geosave_engine.model import ModelChain, Published, build_model, chain_step
from geosave_engine.model.registry import register_model


@register_model("encoder", "tiny")
class Encoder(nn.Module):
    channels: Published[int]

    def __init__(self, channels: int = 16):
        super().__init__()
        self.channels = channels
        self.net = nn.Sequential(nn.Conv2d(4, channels, 3, stride=2, padding=1), nn.ReLU())

    @chain_step()
    def forward(self, image: torch.Tensor) -> torch.Tensor:
        features = self.net(image)
        return features


@register_model("decoder", "upsample")
class Decoder(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.net = nn.Sequential(nn.Upsample(scale_factor=2), nn.Conv2d(channels, channels, 3, padding=1), nn.ReLU())

    @chain_step()
    def forward(self, features: torch.Tensor) -> torch.Tensor:
        feature_map = self.net(features)
        return feature_map


@register_model("head", "pixel")
class Head(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.net = nn.Conv2d(channels, 10, 1)

    @chain_step(head=True)
    def forward(self, feature_map: torch.Tensor) -> torch.Tensor:
        return self.net(feature_map)

In [ ]:
model = nn.Sequential(Encoder(), Decoder(channels=16), Head(channels=16))
model

## Connect the parts by what they read and return

`nn.Sequential` works well when each `forward` takes the previous part's result. Other models need richer interfaces: an encoder might return several feature maps, or a decoder might need both encoded features and the original image. Without a shared interface, swapping either part can mean writing new glue for that encoder–decoder pair.

`chain_step` lets each part declare the named values it needs and produces. When an encoder and several decoders share that interface, the encoder can support all of them without a separate adapter for each combination. Each part describes its own interface once.

`ModelChain` connects these inputs and outputs into a dependency graph. Here, the encoder produces `features`, the decoder turns them into `feature_map`, and the head returns the final scores. Displaying the model shows that data flow above its layers.

In [ ]:
model = ModelChain(
    encoder=Encoder(),
    decoder=Decoder(channels=16),
    head=Head(channels=16),
)
model

## Swap the decoder

This decoder resizes the features to the original image's dimensions. Its `forward` therefore needs two inputs: `features` from the encoder and `image` from the caller. Swap it into the same chain:

In [ ]:
class ResizeDecoder(nn.Module):
    @chain_step()
    def forward(self, features: torch.Tensor, image: torch.Tensor) -> torch.Tensor:
        feature_map = nn.functional.interpolate(
            features, size=image.shape[-2:], mode="bilinear", align_corners=False
        )
        return feature_map


model = ModelChain(
    encoder=Encoder(),
    decoder=ResizeDecoder(),
    head=Head(channels=16),
)
model

The encoder and head need no changes. `ModelChain` keeps the caller's inputs and named outputs from earlier steps available throughout the forward call. Each step receives only the values it asks for, so this decoder can still receive `image` after the encoder has produced `features`. The caller still uses `model(image=image)`.

This works with normal PyTorch training. Here, the decoder reads only the image's dimensions; gradients flow through the resized features back to the encoder. Passing a tensor to another step preserves its gradient connection. We do not detach the features, because we want the encoder to learn from the final loss.

## Change the width in one place

We still had to repeat the channel count when constructing the parts. The encoder's class attribute `channels: Published[int]` makes its width available to parts built after it. The decoder and head already take `channels`, so they can receive that value without knowing which encoder supplied it.

`build_model` constructs the parts in order and passes along published values. Give it the classes and it builds the same model using the encoder's default width:

In [ ]:
model = build_model({"encoder": Encoder, "decoder": Decoder, "head": Head})
model

Now widening the encoder is one change. Give it a module built with 32 channels; the decoder and head follow on their own. The model's displayed layers show the new widths.

In [ ]:
model = build_model({"encoder": Encoder(channels=32), "decoder": Decoder, "head": Head})
model

## Build your own registered model

The `@register_model` decorators on our classes give each implementation a name within its stage: `tiny`, `upsample`, and `pixel`. We can select these custom parts by name just as we would select a built-in model.

Here is the same wider model as a construction recipe. Only the encoder needs `channels` in `init_args`; the decoder and head still receive its published width. Because this recipe is plain data, it can also live in a YAML or JSON config.

In [ ]:
model = build_model({
    "encoder": {"name": "tiny", "init_args": {"channels": 32}},
    "decoder": {"name": "upsample"},
    "head": {"name": "pixel"},
})
model

## Load a built-in model

Our registered classes join the same catalogue as GeoSave's built-in models. It includes DINOv3, Prithvi and Clay encoders, DPT and U-Net decoders, and heads for segmentation, classification, regression and detection. We can see the available names by stage:


In [ ]:
from geosave_engine.model.registry import list_models

list_models()

Let's load [Prithvi-EO-2.0 Tiny TL](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-tiny-TL), the smallest built-in encoder with published pretrained weights, with about 5.6 million encoder parameters. `pretrained=True` downloads those weights on the first run and reuses the cache afterward.

We pair it with a U-Net decoder and a new segmentation head for our ten classes, named in the dataset's `classes.yaml`. The encoder publishes its feature widths and strides; the decoder and head receive the values they need through `build_model`, just as our custom parts did. Only the encoder starts with pretrained weights; the decoder and head need training for the task.


In [ ]:
import yaml

legend = yaml.safe_load((root / "classes.yaml").read_text())
classes = list(legend["class_map"].values())

pretrained_model = build_model({
    "encoder": {
        "name": "prithvi_tl",
        "init_args": {
            "model_name": "prithvi_eo_v2_tiny_tl",
            "pretrained": True,
            "bands": ["BLUE", "GREEN", "RED", "NIR_NARROW"],
        },
    },
    "decoder": {"name": "unet"},
    "head": {
        "name": "segmentation",
        "init_args": {"classes": classes},
    },
})
pretrained_model

The graph now starts with `image`, `temporal_coords` and `location_coords`: this encoder uses the acquisition date and geographic location as well as the pixels.

It was pretrained on six bands: blue, green, red, narrow near-infrared and two shortwave-infrared bands. Our tiles carry four of them, so `bands` names those four in the order our tensors hold them. Their pretrained weights carry over and the two shortwave-infrared ones are dropped; the rest of the encoder is pretrained either way. Our `B08` is Sentinel-2's broad near-infrared band, close to the narrow one Prithvi learned from.

The encoder still needs each tile's date and location, and inputs prepared the way it was trained, so we keep training the custom `model` below. Notebook 2 prepares better inputs, and notebook 3 fetches imagery with the dates and places an encoder like this reads.


## Augmentation

200 tiles is little data for a segmentation model. Flipping each batch at random shows the model a different view every step; `ImageAugmenter` applies the same flip to the image and its labels so they stay matched.

Each augmentation names a Kornia transform and supplies its arguments under `init_args`. The training step below adds a channel axis to the label mask for augmentation, then removes it before calculating the loss. The rest is the same Lightning training setup as notebook 0, now using our registered model.

In [ ]:
import lightning as L
from lightning.pytorch.utilities.model_summary.model_summary import ModelSummary
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex

from geosave_engine.ml.transforms import ImageAugmenter

augmentations = [
    {"name": "RandomHorizontalFlip", "init_args": {"p": 0.5}},
    {"name": "RandomVerticalFlip", "init_args": {"p": 0.5}},
]


class Segmenter(L.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.example_input_array = torch.zeros(1, 4, 510, 510)
        metrics = MetricCollection({
            "miou": MulticlassJaccardIndex(num_classes=10, ignore_index=255),
            "accuracy": MulticlassAccuracy(num_classes=10, ignore_index=255, average="micro"),
        })
        self.val_metrics = metrics.clone(prefix="val_")
        self.augment = ImageAugmenter(augmentations, size=510)

    def forward(self, image):
        return self.model(image=image)

    def training_step(self, batch, batch_index):
        image, label = batch
        image, label = self.augment(image, label[:, None].float(), data_keys=["input", "mask"])
        label = label[:, 0].long()
        loss = nn.functional.cross_entropy(self(image), label, ignore_index=255)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_index):
        image, label = batch
        self.val_metrics(self(image), label)
        self.log_dict(self.val_metrics)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-2)


module = Segmenter(model)
ModelSummary(module, max_depth=-1)

In [ ]:
trainer = L.Trainer(max_epochs=3)
trainer.fit(module, train_loader, val_loader)

## Compare the runs

Notebook 0 and this notebook both log to `lightning_logs/` next to them, one version per run. TensorBoard draws them on the same axes, so we can compare validation accuracy and mIoU. This run trains longer than notebook 0's short demonstration; keep that difference in mind when reading the curves.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

## What happened

The model's parts can now be resized or swapped one at a time, and our own implementations can be selected by name. The same recipe also loaded a pretrained encoder with a built-in decoder and head. For our training run, we widened the custom encoder and added flips; the curves show what that run measured.

The model we trained still reads four numbers per pixel without knowing what they are. The next notebook looks at what the satellite recorded and what each picture knows about itself.

| Next | Question |
| --- | --- |
| 2 · Inputs | What do the four numbers mean, and how do I give the model better inputs? |